# EMG Segraw → Moments Feature Engineering

Pipeline:
1. Load raw segmented EMG CSVs into nested dict
2. BPF at 20-450 Hz
3. Per-gesture std normalisation
4. Khushaba spectral moments FE (5 features × 16 EMG channels = 80 features per gesture)
5. Global feature-matrix std normalisation
6. Save to CSV — one row per gesture trial

Mirror of `ppdsegraw_feature_engineering.ipynb` for EMG, with two changes:
- `modalities=['E']` in `load_segraw_data`

Output file: `moments_segraw_allEMG.csv`

In [1]:
import numpy as np
import pandas as pd
import time
import json

# All shared preprocessing + FE functions live here
from shared_processing import (
    load_segraw_data,
    apply_filter_to_nested_dict,
    normalize_gestures_by_std_any_channels,
    create_khushaba_spectralmomentsFE_vectors,
    normalize_whole_dataset_features,
)

## 0. Config — set your paths here

In [2]:
# ── UPDATE THESE TO YOUR MACHINE ──────────────────────────────────────────
# C:\Users\kdmen\Box\Yamagami Lab\Data\2024_UIST_dataset\upload\segmented_raw_data
RAW_DATA_DIR   = r"C:\Users\kdmen\Box\Yamagami Lab\Data\2024_UIST_dataset\upload\segmented_raw_data"
SAVE_DIR       = r"C:\Users\kdmen\Box\Yamagami Lab\Data\Meta_Gesture_Project"
# ──────────────────────────────────────────────────────────────────────────

# Intermediate save (nested dict after loading, before FE) — optional but
# saves ~5 min if you need to re-run FE without reloading CSVs
RAW_DICT_SAVE_PATH      = SAVE_DIR + r"\segraw0_EMG_allgestures_allusers.json"
PPD_DICT_SAVE_PATH      = SAVE_DIR + r"\ppdsegraw1_allEMG.json"
MOMENTS_CSV_SAVE_PATH   = SAVE_DIR + r"\moments_ppdsegraw2_allEMG.csv"

# Participant lists — identical to your EMG pipeline
pIDs_impaired   = ['P102','P103','P104','P105','P106','P107','P108','P109','P110','P111',
                   'P112','P114','P115','P116','P118','P119','P121','P122','P123','P124',
                   'P125','P126','P127','P128','P131','P132']
pIDs_unimpaired = ['P004','P005','P006','P008','P010','P011']
pIDs_both       = pIDs_impaired + pIDs_unimpaired

# NOTE: 2000 Hz was for EMG
# 148 Hz was for IMU
FS = 2000  # Nominal sampling rate stored in the raw files (used for moment calculations)

## 1. Load raw EMG CSVs

> Same `load_segraw_data` as the EMG pipeline
> Each gesture ends up as a list of 16 channels × ~6000 timepoints.

In [ ]:
# 53 mins for exp def

print("Loading raw EMG data...")
start = time.time()

nested_dict = load_segraw_data(
    pIDs      = pIDs_both,
    data_dir_path = RAW_DATA_DIR,
    modalities    = ["E"],          
    expt_types    = ["experimenter-defined"],
    num_emg_channels = 16,
)

print(f"Done in {time.time()-start:.1f}s")
print(f"Participants loaded: {list(nested_dict.keys())}")

Loading raw EMG data...
P102
P103
P104
P105
P106
P107
P108
P109
P110
P111
P112
P114
P115
P116
P118
P119
P121
P122
P123
P124
P125
P126
P127
P128
P131
P132
P004
P005
P006
P008
P010
P011
Done in 3204.6s
Participants loaded: ['P102', 'P103', 'P104', 'P105', 'P106', 'P107', 'P108', 'P109', 'P110', 'P111', 'P112', 'P114', 'P115', 'P116', 'P118', 'P119', 'P121', 'P122', 'P123', 'P124', 'P125', 'P126', 'P127', 'P128', 'P131', 'P132', 'P004', 'P005', 'P006', 'P008', 'P010', 'P011']


In [4]:
# Quick sanity check — should be 72 channels (12 sensors × 6 axes) --> It might think there's 15 sensors since the highest sensor channel was labeled 15...
sample_pid     = list(nested_dict.keys())[0]
sample_gesture = list(nested_dict[sample_pid].keys())[0]
sample_trial   = list(nested_dict[sample_pid][sample_gesture].keys())[0]
sample_data    = nested_dict[sample_pid][sample_gesture][sample_trial]["EMG"]

print(f"Sample: {sample_pid} / {sample_gesture} / trial {sample_trial}")
print(f"  n_channels  : {len(sample_data)}   (expected 16)")
print(f"  n_timepoints: {len(sample_data[0])}  (expected ~6000)")

Sample: P102 / pan / trial 1
  n_channels  : 16   (expected 16)
  n_timepoints: 6380  (expected ~6000)


In [5]:
# Optional: save the raw nested dict so you don't have to reload CSVs next time
print("Saving raw EMG nested dict...")
import json
with open(RAW_DICT_SAVE_PATH, 'w') as f:
    json.dump(nested_dict, f)
print(f"Saved → {RAW_DICT_SAVE_PATH}")

Saving raw EMG nested dict...
Saved → C:\Users\kdmen\Box\Yamagami Lab\Data\Meta_Gesture_Project\segraw0_EMG_allgestures_allusers.json


## 2. Normalise and BPF, mean-subtract, then per-gesture std=1

> We do mean subtraction + per-gesture std normalisation and BPFing

In [6]:
start = time.time()

ms_dict = apply_filter_to_nested_dict(
    nested_dict,
    normalization_method = "MEANSUBTRACTION",
    already_BPFd         = False,  
)

print(f"Done in {time.time()-start:.1f}s")

Done in 78.9s


In [7]:
print("Per-gesture std normalisation...")
start = time.time()

# Use the any-channels variant so it handles 16 EMG channels automatically
ppd_dict = normalize_gestures_by_std_any_channels(ms_dict)

print(f"Done in {time.time()-start:.1f}s")

# Sanity check: std across flattened channels should be ~1
sample_data_ppd = ppd_dict[sample_pid][sample_gesture][sample_trial]["EMG"]
flat = [v for ch in sample_data_ppd for v in ch]
print(f"  Post-normalisation std (should be ~1.0): {np.std(flat):.4f}")

Per-gesture std normalisation...
Done in 390.3s
  Post-normalisation std (should be ~1.0): 1.0000


In [8]:
# Optional: save the preprocessed dict
print("Saving preprocessed EMG dict...")
with open(PPD_DICT_SAVE_PATH, 'w') as f:
    json.dump(ppd_dict, f)
print(f"Saved → {PPD_DICT_SAVE_PATH}")

Saving preprocessed EMG dict...
Saved → C:\Users\kdmen\Box\Yamagami Lab\Data\Meta_Gesture_Project\ppdsegraw1_allEMG.json


## 3. Khushaba Moments Feature Engineering

For each gesture trial, for each of the 16 EMG channels, we compute:
- `zero_order`        — total signal power (energy)
- `second_order`      — energy of rate-of-change (first derivative)
- `fourth_order`      — energy of jerk (second derivative)
- `sparsity`          — sqrt(m0*m4) / m2
- `irregularity_factor` — m2^2 / (m0*m4), normalised by waveform length

Result: **80 features per gesture trial** (16 channels × 5 features).
All degenerate values (e.g. near-constant channels) are handled by
`safe_log` — clipped to [-10, 10] rather than blowing up to ±inf.

In [9]:
print("Computing Khushaba moments...")
start = time.time()

moments_df = create_khushaba_spectralmomentsFE_vectors(ppd_dict, fs=FS)

print(f"Done in {time.time()-start:.1f}s")
print(f"Shape: {moments_df.shape}   (expected: n_trials × 453 = n_trials × [3 meta + 450 features])")
moments_df.head()

Computing Khushaba moments...
Done in 30.9s
Shape: (3200, 83)   (expected: n_trials × 453 = n_trials × [3 meta + 450 features])


,Participant,Gesture_ID,Gesture_Num,EMG0_zero_order,EMG0_second_order,EMG0_fourth_order,EMG0_sparsity,EMG0_irregularity_factor,EMG1_zero_order,EMG1_second_order,...,EMG14_zero_order,EMG14_second_order,EMG14_fourth_order,EMG14_sparsity,EMG14_irregularity_factor,EMG15_zero_order,EMG15_second_order,EMG15_fourth_order,EMG15_sparsity,EMG15_irregularity_factor
0,P102,pan,1,-0.831166,-1.489677,-1.040459,0.553865,0.755973,0.256936,-2.384009,...,-1.857505,0.156021,2.858352,0.344403,1.439974,-1.695468,-0.706756,1.665808,0.691926,1.166405
1,P102,pan,2,-0.152417,-2.128188,-3.060945,0.521507,0.430267,1.084921,-3.259037,...,-1.464002,-0.264959,1.649828,0.357872,1.155465,-1.256452,-0.972152,0.488133,0.587992,0.959295
2,P102,pan,3,-0.423968,-1.982067,-2.384248,0.577959,0.580875,0.620104,-2.916419,...,-1.731388,0.005615,2.456031,0.356706,1.367943,-1.802879,-0.445942,2.180683,0.634844,1.156030
3,P102,pan,4,-0.258082,-2.033692,-2.798878,0.505212,0.591960,0.566743,-2.777438,...,-1.695160,0.022586,2.393700,0.326685,1.410289,-1.885480,-0.361120,2.438832,0.637796,1.240141
4,P102,pan,5,-0.133329,-2.215566,-3.281228,0.508288,0.527234,0.311004,-2.450629,...,-1.630410,-0.096158,2.117578,0.339742,1.358841,-1.635121,-0.701116,1.587064,0.677087,1.078792


In [10]:
# Check for any surviving NaN or inf (should be zero with safe_log)
n_nan = moments_df.isna().sum().sum()
n_inf = np.isinf(moments_df.select_dtypes(include=np.number).values).sum()
print(f"NaN count: {n_nan}  |  Inf count: {n_inf}")

if n_nan > 0 or n_inf > 0:
    print("WARNING: Unexpected NaN/inf — imputing with column mean as fallback.")
    moments_df.replace([np.inf, -np.inf], np.nan, inplace=True)
    feature_cols = [c for c in moments_df.columns if c not in ['Participant','Gesture_ID','Gesture_Num']]
    moments_df[feature_cols] = moments_df[feature_cols].fillna(moments_df[feature_cols].mean())
else:
    print("Clean — no NaN or inf values.")

NaN count: 0  |  Inf count: 0
Clean — no NaN or inf values.


## 4. Global feature-matrix normalisation

Divides every feature by the global standard deviation of the entire
feature matrix, so all 450 features are on the same scale.
This is the same final step as in the EMG pipeline.

In [11]:
n_moments_df = normalize_whole_dataset_features(moments_df)

print(f"Shape after normalisation: {n_moments_df.shape}")
n_moments_df.head()

Shape after normalisation: (3200, 83)


,Participant,Gesture_ID,Gesture_Num,EMG0_zero_order,EMG0_second_order,EMG0_fourth_order,EMG0_sparsity,EMG0_irregularity_factor,EMG1_zero_order,EMG1_second_order,...,EMG14_zero_order,EMG14_second_order,EMG14_fourth_order,EMG14_sparsity,EMG14_irregularity_factor,EMG15_zero_order,EMG15_second_order,EMG15_fourth_order,EMG15_sparsity,EMG15_irregularity_factor
0,P102,pan,1,-0.226755,-0.406408,-0.283854,0.151103,0.206242,0.070096,-0.650396,...,-0.506758,0.042565,0.779805,0.093959,0.392848,-0.462551,-0.192814,0.454459,0.188769,0.318214
1,P102,pan,2,-0.041582,-0.580604,-0.835076,0.142276,0.117384,0.295984,-0.889118,...,-0.399404,-0.072285,0.450100,0.097633,0.315230,-0.342780,-0.265219,0.133171,0.160414,0.261711
2,P102,pan,3,-0.115665,-0.540740,-0.650462,0.157677,0.158472,0.169175,-0.795647,...,-0.472351,0.001532,0.670045,0.097315,0.373197,-0.491855,-0.121660,0.594926,0.173196,0.315384
3,P102,pan,4,-0.070409,-0.554824,-0.763579,0.137830,0.161496,0.154617,-0.757730,...,-0.462467,0.006162,0.653040,0.089125,0.384750,-0.514390,-0.098519,0.665353,0.174001,0.338331
4,P102,pan,5,-0.036374,-0.604442,-0.895172,0.138669,0.143838,0.084847,-0.668572,...,-0.444802,-0.026233,0.577710,0.092687,0.370714,-0.446088,-0.191276,0.432977,0.184720,0.294312


## 5. Save

In [12]:
n_moments_df.to_csv(MOMENTS_CSV_SAVE_PATH, index=False, header=True)
print(f"Saved → {MOMENTS_CSV_SAVE_PATH}")
print(f"Final shape: {n_moments_df.shape}")
print(f"Columns: {list(n_moments_df.columns[:6])} ... {list(n_moments_df.columns[-3:])}")

Saved → C:\Users\kdmen\Box\Yamagami Lab\Data\Meta_Gesture_Project\moments_ppdsegraw2_allEMG.csv
Final shape: (3200, 83)
Columns: ['Participant', 'Gesture_ID', 'Gesture_Num', 'EMG0_zero_order', 'EMG0_second_order', 'EMG0_fourth_order'] ... ['EMG15_fourth_order', 'EMG15_sparsity', 'EMG15_irregularity_factor']


## 6. Quick verification

In [13]:
# Reload and spot-check
check_df = pd.read_csv(MOMENTS_CSV_SAVE_PATH)
print(f"Reloaded shape : {check_df.shape}")
print(f"Participants   : {sorted(check_df['Participant'].unique())}")
print(f"Gestures       : {sorted(check_df['Gesture_ID'].unique())}")
print(f"Trials per gesture per participant (should be 10):")
print(check_df.groupby(['Participant','Gesture_ID'])['Gesture_Num'].count().describe())

feature_cols = [c for c in check_df.columns if c not in ['Participant','Gesture_ID','Gesture_Num']]
print(f"\nFeature stats (post-normalisation):")
print(check_df[feature_cols].describe().loc[['mean','std','min','max']])

Reloaded shape : (3200, 83)
Participants   : ['P004', 'P005', 'P006', 'P008', 'P010', 'P011', 'P102', 'P103', 'P104', 'P105', 'P106', 'P107', 'P108', 'P109', 'P110', 'P111', 'P112', 'P114', 'P115', 'P116', 'P118', 'P119', 'P121', 'P122', 'P123', 'P124', 'P125', 'P126', 'P127', 'P128', 'P131', 'P132']
Gestures       : ['close', 'delete', 'duplicate', 'move', 'open', 'pan', 'rotate', 'select-single', 'zoom-in', 'zoom-out']
Trials per gesture per participant (should be 10):
count    320.0
mean      10.0
std        0.0
min       10.0
25%       10.0
50%       10.0
75%       10.0
max       10.0
Name: Gesture_Num, dtype: float64

Feature stats (post-normalisation):
      EMG0_zero_order  EMG0_second_order  EMG0_fourth_order  EMG0_sparsity   
mean        -0.338077          -0.337820          -0.205821       0.145637  \
std          0.717593           0.788842           1.819669       0.034788   
min         -2.674436          -1.897830          -2.728163       0.064249   
max          1.080164